# Filtered convection recovery on a Chebyshev-Lobatto grid

This is a test pipeline for the quadrature-enabled PySPIDER fork. Install the exact, unmodified fork revision with:

```bash
python -m pip install --no-deps -r requirements-chebyshev-lobatto.txt
```

The notebook uses periodic filtering in `x` and Clenshaw-Curtis-weighted filtering in `z`. The vertical filter is evaluated directly on a Lobatto grid covering the support-safe interior, so the finite-support boundary region is excluded while PySPIDER can use its existing `clenshaw-curtis` rule. Since PySPIDER accepts one node specification per dataset, every recovery domain spans this vertical grid; `x` and time remain randomly windowed.

In [ ]:
import gc
import os

import jax
import jax.numpy as jnp
import numpy as np
from numpy.polynomial.chebyshev import chebpts2

try:
    from PySPIDER.commons.identify_models import *
    from PySPIDER.commons.integration import SUPPORTED_SCHEMES
    from PySPIDER.commons.quadrature_schemes import clenshaw_curtis_weights
    from PySPIDER.commons.sparse_reg_bf import (
        AIC, Initializer, ModelIterator, Residual, Scaler, Threshold
    )
    from PySPIDER.continuous.library import *
    from PySPIDER.continuous.process_library_terms import *
except (ImportError, ModuleNotFoundError) as exc:
    raise RuntimeError(
        "Install the pinned fork with: python -m pip install --no-deps "
        "-r requirements-chebyshev-lobatto.txt"
    ) from exc

if "clenshaw-curtis" not in SUPPORTED_SCHEMES:
    raise RuntimeError("The installed PySPIDER is not the quadrature-enabled fork")

jax.config.update("jax_enable_x64", True)
print("PySPIDER quadrature schemes:", SUPPORTED_SCHEMES)

In [ ]:
archive_path = "/storage/scratch1/2/ashelestov3/preprocessed_paper_data_lobatto.npz"
print("Loading", archive_path)

with np.load(archive_path) as data:
    # Reverse the historical archive-name swap made by preprocessing.
    V = np.asarray(data["U"])
    U = np.asarray(data["V"])
    P = np.asarray(data["P"])
    B = np.asarray(data["B"])
    x_full = np.asarray(data["x"])
    y_full = np.asarray(data["y"])
    t = np.asarray(data["t"])
    grid_kind = str(data["z_grid_kind"])
    z_interval = np.asarray(data["z_interval"], dtype=float)

assert grid_kind == "chebyshev_lobatto"
assert U.shape == V.shape == P.shape == B.shape
assert U.shape == (x_full.size, y_full.size, t.size)
assert np.all(np.diff(x_full) > 0)
assert np.all(np.diff(y_full) > 0)
assert np.all(np.diff(t) > 0)

expected_y = (
    0.5 * z_interval.sum()
    + 0.5 * np.diff(z_interval)[0] * chebpts2(y_full.size)
)
np.testing.assert_allclose(y_full, expected_y, rtol=0.0, atol=5e-12)

dx_steps = np.diff(x_full)
dt_steps = np.diff(t)
dx = float(np.mean(dx_steps))
dt = float(np.mean(dt_steps))
if not np.allclose(dx_steps, dx, rtol=1e-10, atol=1e-13):
    raise ValueError("The x grid is not uniform")
if not np.allclose(dt_steps, dt, rtol=1e-8, atol=1e-12):
    raise ValueError("The current PySPIDER test expects uniformly sampled times")

Rayleigh = 1e8
Prandtl = 1.0
kappa = (Rayleigh * Prandtl) ** -0.5
print("Shape:", U.shape, "dx:", dx, "dt from np.diff(t):", dt)
print("Nonuniform z spacing range:", np.diff(y_full).min(), np.diff(y_full).max())

In [ ]:
from functools import partial

def _uniform_gaussian_kernel(spacing, delta, alpha):
    sigma = delta / np.sqrt(12.0)
    radius = alpha * delta
    radius_points = int(np.ceil(radius / spacing))
    offsets = np.arange(-radius_points, radius_points + 1) * spacing
    kernel = np.exp(-0.5 * (offsets / sigma) ** 2)
    return jnp.asarray(kernel / kernel.sum())

@partial(jax.jit, static_argnums=(2,))
def _periodic_convolve_1d(array, kernel, axis):
    radius_points = kernel.size // 2
    pad = [(0, 0)] * array.ndim
    pad[axis] = (radius_points, radius_points)
    padded = jnp.pad(array, pad, mode="wrap")
    moved = jnp.moveaxis(padded, axis, 0)
    flat = moved.reshape((moved.shape[0], -1)).T
    convolved = jax.vmap(lambda line: jnp.convolve(line, kernel, mode="valid"))(flat)
    restored = convolved.T.reshape((array.shape[axis],) + moved.shape[1:])
    return jnp.moveaxis(restored, 0, axis)

def _lobatto_gaussian_operator(source_coordinates, target_coordinates, delta, alpha):
    """Finite-support Gaussian convolution from Lobatto source to target nodes."""
    source_coordinates = np.asarray(source_coordinates, dtype=float)
    target_coordinates = np.asarray(target_coordinates, dtype=float)
    sigma = delta / np.sqrt(12.0)
    radius = alpha * delta
    quadrature = clenshaw_curtis_weights(
        source_coordinates.size - 1,
        source_coordinates[0],
        source_coordinates[-1],
    )
    distances = target_coordinates[:, None] - source_coordinates[None, :]
    kernel = np.exp(-0.5 * (distances / sigma) ** 2)
    kernel[np.abs(distances) > radius] = 0.0
    weighted = kernel * quadrature[None, :]
    normalization = weighted.sum(axis=1, keepdims=True)
    if np.any(normalization <= 0):
        raise ValueError("A vertical Gaussian stencil has zero quadrature mass")
    return jnp.asarray(weighted / normalization)

delta = 50.0 * dx
alpha = 3.0
support_radius = alpha * delta
x_kernel = _uniform_gaussian_kernel(dx, delta, alpha)

safe_z_min = y_full[0] + support_radius
safe_z_max = y_full[-1] - support_radius
target_nz = int(np.count_nonzero((y_full >= safe_z_min) & (y_full <= safe_z_max)))
if target_nz < 2 or safe_z_min >= safe_z_max:
    raise ValueError("The requested Gaussian support removes the entire z domain")

# Evaluate the filtered fields directly on a complete mapped Lobatto grid in
# the support-safe interior. This avoids a truncated-grid moment-matching solve.
y = 0.5 * (safe_z_min + safe_z_max) + 0.5 * (safe_z_max - safe_z_min) * chebpts2(target_nz)
z_operator = _lobatto_gaussian_operator(y_full, y, delta, alpha)
x = x_full.copy()  # periodic x is never trimmed

@jax.jit
def _filter_field(field):
    x_filtered = _periodic_convolve_1d(field, x_kernel, axis=0)
    return jnp.einsum("ij,xjt->xit", z_operator, x_filtered, optimize=True)

def filter_to_lobatto_interior(field, label):
    print("Filtering", label)
    filtered = np.asarray(_filter_field(jnp.asarray(field)))
    gc.collect()
    return filtered

print("Filter width:", delta, "support radius:", support_radius)
print("Target Lobatto z points:", y.size, "range:", (y[0], y[-1]))

In [ ]:
# Primary filtered fields.
u_filt = filter_to_lobatto_interior(U, "u")
v_filt = filter_to_lobatto_interior(V, "v")
p_filt = filter_to_lobatto_interior(P, "p")  # ordinary filtered pressure
b_filt = filter_to_lobatto_interior(B, "b")

# Full subfilter stress tau_ij = overline(u_i u_j) - ubar_i ubar_j.
tau_xx = filter_to_lobatto_interior(U * U, "u*u") - u_filt * u_filt
tau_zz = filter_to_lobatto_interior(V * V, "v*v") - v_filt * v_filt
tau_xz = filter_to_lobatto_interior(U * V, "u*v") - u_filt * v_filt

u_bar = np.stack((u_filt, v_filt), axis=-1)
tau = np.empty(u_filt.shape + (2, 2), dtype=u_filt.dtype)
tau[..., 0, 0] = tau_xx
tau[..., 0, 1] = tau_xz
tau[..., 1, 0] = tau_xz
tau[..., 1, 1] = tau_zz

# Subfilter buoyancy flux phi_i = overline(u_i b) - ubar_i bbar.
phi_x = filter_to_lobatto_interior(U * B, "u*b") - u_filt * b_filt
phi_z = filter_to_lobatto_interior(V * B, "v*b") - v_filt * b_filt
phi = np.stack((phi_x, phi_z), axis=-1)

del U, V, P, B, u_filt, v_filt, tau_xx, tau_zz, tau_xz, phi_x, phi_z
gc.collect()

print("u_bar:", u_bar.shape)
print("p, b:", p_filt.shape, b_filt.shape)
print("tau:", tau.shape, "phi:", phi.shape)

In [ ]:
# Build the dataset with the fork's public nonuniform-grid and quadrature API.
unit_z = np.stack((np.zeros_like(b_filt), np.ones_like(b_filt)), axis=-1)

uobs = Observable(string="u", rank=1)
pobs = Observable(string="p", rank=0)
bobs = Observable(string="b", rank=0)
zobs = Observable(string="z̄", rank=1)
Tobs = Observable(string="T", rank=2, can_commute_indices=True, antisymmetric=False)
fobs = Observable(string="ϕ", rank=1)
observables = [uobs, bobs, pobs, zobs, Tobs, fobs]  # vorticity w intentionally omitted

data_dict = {
    "u": u_bar,
    "b": b_filt,
    "p": p_filt,
    "z̄": unit_z,
    "T": tau,
    "ϕ": phi,
}

world_size = np.asarray(b_filt.shape, dtype=int)
quadrature = {
    1: {
        "clenshaw-curtis": {
            "nodes": y,
            "interval": [float(y[0]), float(y[-1])],
        }
    }
}

np.random.seed(1)
srd = SRDataset(
    world_size=world_size,
    data_dict=data_dict,
    observables=observables,
    dxs=[dx, float(np.mean(np.diff(y))), dt],
    grids=[x, y, t],
    schemes_and_options=quadrature,
    irreps=SRDataset.all_rank2_irreps(),
)
srd.make_libraries(max_complexity=3)

# Remove symbolic identities caused by the constant vertical unit vector.
remove_terms = {
    "z̄_β · ∂α z̄_β",
    "∂t b · z̄_α",
    "z̄_α · z̄_β · z̄_β",
    "z̄_β · z̄_β · ϕ_α",
    "∂α z̄_β",
}
for irrep in srd.irreps:
    lib = srd.libs[irrep]
    before = len(lib.terms)
    lib.terms = [term for term in lib.terms if str(term) not in remove_terms]
    print(irrep, "removed", before - len(lib.terms), "symbolic terms;", len(lib.terms), "remain")

In [ ]:
# Test-run defaults. Increase SPIDER_TEST_NDOMAINS after validating the path.
dom_x = 64
dom_z = int(world_size[1])  # fixed nodes require every domain to span the target Lobatto grid
dom_time = 40
ndomains = int(os.environ.get("SPIDER_TEST_NDOMAINS", "32"))

if dom_x > world_size[0] or dom_time > world_size[2]:
    raise ValueError("Requested test domain is larger than the available data")

available_cores = os.cpu_count() or 1
num_processors = min(ndomains, max(1, available_cores - 4))
srd.make_domains(
    ndomains=ndomains,
    domain_size=[dom_x, dom_z, dom_time],
    pad=0,
)
srd.make_weights(m=16, qmax=0)

L = (dom_x - 1) * dx
T = (dom_time - 1) * dt  # no obsolete factor of 10
srd.set_LT_scale(L=L, T=T)
print("Domains:", ndomains, "shape:", srd.domain_size, "workers:", num_processors)
print("Characteristic L, T:", L, T)
srd.make_library_matrices(debug=False, parallel=True, num_processors=num_processors)

In [ ]:
def prune_numerical_library_columns(lib, zero_rtol=1e-12, duplicate_rtol=1e-10):
    """Remove numerically zero and equal/opposite duplicate Q columns."""
    Q = np.asarray(lib.Q)
    norms = np.linalg.norm(Q, axis=0)
    scale = norms.max(initial=0.0)
    zero = norms <= zero_rtol * scale

    keep = []
    duplicates = []
    normalized = np.zeros_like(Q, dtype=float)
    nonzero = ~zero
    normalized[:, nonzero] = Q[:, nonzero] / norms[nonzero]

    tolerance = duplicate_rtol * np.sqrt(max(1, Q.shape[0]))
    for column in np.flatnonzero(nonzero):
        vector = normalized[:, column]
        duplicate_of = None
        for previous in keep:
            if min(
                np.linalg.norm(vector - normalized[:, previous]),
                np.linalg.norm(vector + normalized[:, previous]),
            ) <= tolerance:
                duplicate_of = previous
                break
        if duplicate_of is None:
            keep.append(column)
        else:
            duplicates.append((column, duplicate_of))

    keep = np.asarray(keep, dtype=int)
    old_terms = list(lib.terms)
    lib.Q = Q[:, keep]
    lib.terms = [old_terms[index] for index in keep]
    if lib.col_weights is not None:
        weights = np.asarray(lib.col_weights)
        lib.col_weights = list(weights[keep])

    print(
        lib.irrep,
        "removed", int(zero.sum()), "zero and", len(duplicates),
        "duplicate columns;", len(keep), "remain",
    )
    return {
        "zero_terms": [old_terms[index] for index in np.flatnonzero(zero)],
        "duplicate_terms": [
            (old_terms[index], old_terms[previous]) for index, previous in duplicates
        ],
    }

pruning_report = {
    irrep: prune_numerical_library_columns(srd.libs[irrep])
    for irrep in srd.irreps
}

In [ ]:
# Evaluate the known filtered momentum balance directly before regression:
# dt(u) + u.grad(u) + grad(p) - kappa*lap(u) - b*z + div(tau) = 0.
momentum_coefficients = {
    "∂t u_α": 1.0,
    "u_β · ∂β u_α": 1.0,
    "∂α p": 1.0,
    "∂β² u_α": -kappa,
    "b · z̄_α": -1.0,
    "∂β T_αβ": 1.0,
}

vector_irrep = next(irrep for irrep in srd.irreps if str(irrep) == "Rank 1")
vector_lib = srd.libs[vector_irrep]
term_index = {str(term): index for index, term in enumerate(vector_lib.terms)}
missing = [name for name in momentum_coefficients if name not in term_index]
if missing:
    raise KeyError(f"Known momentum terms missing after pruning: {missing}")

known_columns = []
known_residual = np.zeros(vector_lib.Q.shape[0])
for name, coefficient in momentum_coefficients.items():
    contribution = coefficient * vector_lib.Q[:, term_index[name]]
    known_columns.append(contribution)
    known_residual += contribution

known_denominator = sum(np.linalg.norm(column) for column in known_columns)
known_relative_residual = np.linalg.norm(known_residual) / known_denominator
print("Known filtered momentum relative balance residual:", known_relative_residual)
print("Known coefficients:", momentum_coefficients)

In [ ]:
# Sparse recovery follows only after the direct known-balance check.
reg_opts_list = []
for irrep in srd.irreps:
    lib = srd.libs[irrep]
    scaler = Scaler(
        sub_inds=None,
        char_sizes=lib.col_weights,
        row_norms=None,
        unit_rows=True,
        train_fraction=0.8,
    )
    reg_opts_list.append({
        "scaler": scaler,
        "initializer": Initializer(method="power", start_k=min(10, len(lib.terms))),
        "residual": Residual(residual_type="matrix_relative"),
        "model_iterator": ModelIterator(max_k=min(10, len(lib.terms)), backward_forward=True, max_passes=3),
        "threshold": Threshold(threshold_type="jump", gamma=1.5, delta=1e-8, n_terms=None),
        "verbose": False,
        "inhomog": False,
        "inhomog_col": None,
    })

eqs, lambdas, reg_results, derived_eqs, excluded_terms = interleave_identify(
    [srd.libs[irrep] for irrep in srd.irreps],
    reg_opts_list,
    threshold=1e-6,
    report_accuracy=True,
)